# ĐỒ ÁN OLIST E-COMMERCE ANALYTICS — PostgreSQL/Supabase + Google Colab

**Mục tiêu:** chạy 14 BUSINESS QUESTIONS bằng PostgreSQL; phân tích trong Colab; huấn luyện (1) mô hình phân loại khách hàng có mua lại, (2) dự báo doanh thu theo tháng, (3) phân nhóm khách hàng bằng RFM + K-Means.

**Lưu ý quan trọng:** Notebook chưa kết nối và chưa chạy trên cơ sở dữ liệu Supabase của bạn. Có thêm bước upload CSV và tùy chọn nạp CSV lên Supabase; mặc định chức năng ghi dữ liệu đang tắt để tránh ghi đè. Mọi bảng/kết quả chỉ được xác nhận sau khi bạn chạy các ô. Dataset Olist chuẩn thường có 9 bảng với tên `olist_*_dataset`; nếu tên/cột của bạn khác, chạy ô kiểm tra schema và sửa theo kết quả.

**Cách dùng:** chạy từng ô từ trên xuống. Không dán mật khẩu database trực tiếp vào notebook đã chia sẻ.

## 0. Cài thư viện
Chạy một lần mỗi khi khởi động runtime Colab.

In [ ]:
%pip -q install "sqlalchemy>=2.0" "psycopg[binary]" pandas numpy matplotlib seaborn scikit-learn plotly

## 1. Kết nối an toàn tới Supabase PostgreSQL
Trong Supabase mở **Project Settings → Database → Connection string**. Lấy host, database, user, password và port. Với kết nối trực tiếp dùng port thường là `5432`; nếu mạng Colab không kết nối được, thử **Session pooler** theo thông tin Supabase cung cấp (không tự đoán host/user/port).

Không đưa password vào cell hoặc commit lên GitHub. Cell dưới đây hỏi mật khẩu riêng khi chạy.

In [ ]:
import os, getpass, urllib.parse
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

DB_HOST = input("Supabase host (chỉ hostname, không có https://): ").strip()
DB_PORT = input("Port [5432]: ").strip() or "5432"
DB_NAME = input("Database [postgres]: ").strip() or "postgres"
DB_USER = input("Database user: ").strip()
DB_PASSWORD = getpass.getpass("Database password (không hiển thị): ")

# URL encode thông tin đăng nhập để tránh lỗi với ký tự đặc biệt.
url = (f"postgresql+psycopg://{urllib.parse.quote_plus(DB_USER)}:"
       f"{urllib.parse.quote_plus(DB_PASSWORD)}@{DB_HOST}:{DB_PORT}/"
       f"{urllib.parse.quote_plus(DB_NAME)}?sslmode=require")
engine = create_engine(url, pool_pre_ping=True, connect_args={"connect_timeout": 15})
with engine.connect() as conn:
    print(conn.execute(text("SELECT current_database(), current_user, version() ")).fetchone())
print("Kết nối thành công. Nếu có lỗi, chưa chạy các phần truy vấn/ML.")

## 1A. Tải dữ liệu Olist CSV vào Google Colab

Cell này cho phép tải lên **một file ZIP chứa các CSV** hoặc chọn nhiều file CSV trực tiếp. Có thể dùng bộ CSV Olist chuẩn gồm customers, geolocation, order items, payments, reviews, orders, products, sellers và category translation. Nếu ZIP có thư mục con, notebook sẽ tìm CSV bên trong.

> File `Olist data.sql` bạn gửi là script truy vấn/làm sạch theo cú pháp SQL Server; nó không chứa bản ghi dữ liệu CSV. Bạn cần tải bộ CSV Olist riêng (hoặc dùng các CSV đang có). Không cần đưa file `.sql` vào ô upload này.


In [ ]:
from pathlib import Path
from google.colab import files
import zipfile, shutil, os

DATA_DIR = Path('/content/olist_data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

print('Chọn file ZIP chứa CSV Olist hoặc chọn nhiều file .csv. Có thể bấm Cancel nếu đã có dữ liệu trong /content/olist_data.')
uploaded = files.upload()
for uploaded_name in uploaded:
    src = Path('/content') / uploaded_name
    if uploaded_name.lower().endswith('.zip'):
        with zipfile.ZipFile(src) as zf:
            zf.extractall(DATA_DIR)
        print(f'Đã giải nén: {uploaded_name}')
    elif uploaded_name.lower().endswith('.csv'):
        shutil.copy2(src, DATA_DIR / uploaded_name)
        print(f'Đã nhận CSV: {uploaded_name}')
    else:
        print(f'Bỏ qua file không phải CSV/ZIP: {uploaded_name}')

found_csv = sorted(DATA_DIR.rglob('*.csv'))
print(f'\nTìm thấy {len(found_csv)} file CSV trong {DATA_DIR}')
for p in found_csv:
    print(f'- {p.relative_to(DATA_DIR)} ({p.stat().st_size/1024/1024:.1f} MB)')


## 1B. Đọc và kiểm tra các CSV

Notebook dùng tên bảng gốc theo `Olist data.sql`. Các cột ngày được chuyển sang datetime để PostgreSQL/Colab có thể phân tích thời gian. Geolocation có thể rất lớn, vì vậy mặc định chỉ đọc/nạp khi bạn chủ động bật.


In [ ]:
import pandas as pd
from pathlib import Path

# Ánh xạ tên CSV gốc -> tên bảng PostgreSQL mà Olist data.sql sử dụng.
CSV_TO_TABLE = {
    'olist_customers_dataset.csv': 'olist_customers_dataset',
    'olist_geolocation_dataset.csv': 'olist_geolocation_dataset',
    'olist_order_items_dataset.csv': 'olist_order_items_dataset',
    'olist_order_payments_dataset.csv': 'olist_order_payments_dataset',
    'olist_order_reviews_dataset.csv': 'olist_order_reviews_dataset',
    'olist_orders_dataset.csv': 'olist_orders_dataset',
    'olist_products_dataset.csv': 'olist_products_dataset',
    'olist_sellers_dataset.csv': 'olist_sellers_dataset',
    'product_category_name_translation.csv': 'product_category_name_translation',
}

# Đọc CSV tìm thấy ở mọi thư mục con. Nếu có tên file trùng, dùng file đầu tiên theo đường dẫn.
csv_paths = {}
for p in sorted(DATA_DIR.rglob('*.csv')):
    csv_paths.setdefault(p.name.lower(), p)

missing_core = [f for f in [
    'olist_customers_dataset.csv','olist_order_items_dataset.csv',
    'olist_order_payments_dataset.csv','olist_order_reviews_dataset.csv',
    'olist_orders_dataset.csv','olist_products_dataset.csv','olist_sellers_dataset.csv'
] if f not in csv_paths]
if missing_core:
    raise FileNotFoundError('Thiếu CSV bắt buộc: ' + ', '.join(missing_core) +
        '. Hãy upload ZIP/bộ CSV Olist đúng tên file rồi chạy lại cell này.')

dfs_csv = {}
for filename, table_name in CSV_TO_TABLE.items():
    path = csv_paths.get(filename)
    if path is None:
        print(f'Không có {filename}; bỏ qua bảng {table_name}.')
        continue
    # Olist dùng dấu phẩy và UTF-8; low_memory=False tránh suy luận kiểu không nhất quán.
    df = pd.read_csv(path, low_memory=False, encoding='utf-8')
    # Chuẩn hóa tên cột dạng khoảng trắng, không đổi tên cột gốc Olist.
    df.columns = [str(c).strip() for c in df.columns]
    date_cols = {
        'olist_orders_dataset.csv': ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date','order_delivered_customer_date','order_estimated_delivery_date'],
        'olist_order_items_dataset.csv': ['shipping_limit_date'],
        'olist_order_reviews_dataset.csv': ['review_creation_date','review_answer_timestamp'],
    }.get(filename, [])
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
    dfs_csv[table_name] = df
    print(f'{table_name}: {len(df):,} dòng × {len(df.columns)} cột')
    display(df.head(2))

print('Đã đọc', len(dfs_csv), 'bảng CSV. Dữ liệu hiện nằm trong bộ nhớ Colab, chưa ghi lên Supabase.')


## 1C. (Tùy chọn) Nạp CSV từ Colab lên Supabase

**Mặc định tắt để tránh ghi đè dữ liệu.** Chỉ bật nếu bạn muốn tạo/nạp bảng trong schema `public` của Supabase. Cell sẽ dừng nếu phát hiện bảng trùng tên; để thay thế bảng hiện có, bạn phải đổi `REPLACE_EXISTING_TABLES = True` sau khi đã sao lưu. Thay thế bảng có thể xóa khóa/index hiện có và là thao tác phá hủy dữ liệu.

Geolocation có nhiều dòng và không cần cho hầu hết 14 câu hỏi/ba mô hình ML, nên mặc định không nạp bảng này.


In [ ]:
from sqlalchemy import inspect

IMPORT_TO_SUPABASE = False        # Đổi thành True nếu thực sự muốn nạp lên Supabase
REPLACE_EXISTING_TABLES = False   # Chỉ True khi chấp nhận xóa/thay thế các bảng cùng tên
INCLUDE_GEOLOCATION = False      # Bật nếu cần phân tích bảng geolocation; có thể nạp rất lâu

if IMPORT_TO_SUPABASE:
    tables_to_load = [
        'product_category_name_translation',
        'olist_customers_dataset', 'olist_sellers_dataset', 'olist_products_dataset',
        'olist_orders_dataset', 'olist_order_items_dataset',
        'olist_order_payments_dataset', 'olist_order_reviews_dataset'
    ]
    if INCLUDE_GEOLOCATION and 'olist_geolocation_dataset' in dfs_csv:
        tables_to_load.append('olist_geolocation_dataset')
    tables_to_load = [t for t in tables_to_load if t in dfs_csv]

    existing = set(inspect(engine).get_table_names(schema='public'))
    collisions = sorted(existing.intersection(tables_to_load))
    if collisions and not REPLACE_EXISTING_TABLES:
        raise RuntimeError(
            'Các bảng sau đã tồn tại trong public: ' + ', '.join(collisions) +
            '. Để bảo toàn dữ liệu, notebook đã dừng. Hãy sao lưu trước; nếu muốn thay thế, đặt REPLACE_EXISTING_TABLES=True và chạy lại cell.'
        )

    # Đọc theo thứ tự tiện dụng; chunk nhỏ để giảm bộ nhớ và kích thước batch.
    for table in tables_to_load:
        df = dfs_csv[table]
        if REPLACE_EXISTING_TABLES and table in existing:
            mode = 'replace'
        else:
            mode = 'fail'
        print(f'Đang nạp {table}: {len(df):,} dòng ...')
        df.to_sql(table, engine, schema='public', if_exists=mode, index=False,
                  chunksize=1000, method='multi')
        print(f'Đã nạp {table}.')
    print('Hoàn tất. Hãy chạy lại cell kiểm tra bảng/cột ở phần tiếp theo.')
else:
    print('Chưa nạp lên Supabase. CSV đã có trong dfs_csv để kiểm tra/ phân tích trong Colab.')


## 2. Kiểm tra bảng và cột trước khi chạy phân tích
Nếu một bảng không tồn tại hoặc tên cột khác, hãy sửa tên trong SQL trước khi tiếp tục. Dataset chuẩn thường dùng các bảng `olist_orders_dataset`, `olist_order_items_dataset`, `olist_order_payments_dataset`, `olist_order_reviews_dataset`, `olist_products_dataset`, `olist_customers_dataset`, `olist_sellers_dataset`, `olist_geolocation_dataset`, `product_category_name_translation` (bảng dịch có thể tên khác).

In [ ]:
schema_df = pd.read_sql_query(text("""
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_schema NOT IN ('pg_catalog','information_schema')
  AND table_type = 'BASE TABLE'
ORDER BY table_schema, table_name
"""), engine)
display(schema_df)

# Liệt kê cột của các bảng public để đối chiếu với SQL.
columns_df = pd.read_sql_query(text("""
SELECT table_name, ordinal_position, column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'public'
ORDER BY table_name, ordinal_position
"""), engine)
display(columns_df)

## 3. Phân tích 14 câu hỏi nghiệp vụ (SQL chạy riêng từng phần)

Mỗi câu hỏi được tách thành một mục Markdown và một ô code riêng. Chạy lần lượt từng ô để xem bảng kết quả trong Colab. Các truy vấn dùng PostgreSQL qua kết nối `engine` đã tạo ở phần 1; cần nạp đủ các bảng CSV hoặc kết nối tới Supabase trước khi chạy.

### Q1 — Doanh thu theo tháng

**Mục đích:** dùng kết quả để trả lời câu hỏi nghiệp vụ số 1. Kiểm tra tên bảng/cột nếu schema thực tế khác bộ dữ liệu Olist chuẩn.

In [ ]:
# Câu hỏi 1: Q1 — Doanh thu theo tháng
query_sql = """SELECT date_trunc('month', o.order_purchase_timestamp)::date AS month,
       round(sum(p.payment_value)::numeric, 2) AS revenue_brl
FROM olist_orders_dataset o
JOIN (SELECT order_id, sum(payment_value) payment_value
      FROM olist_order_payments_dataset GROUP BY order_id) p USING(order_id)
WHERE o.order_status NOT IN ('canceled','unavailable')
  AND o.order_purchase_timestamp IS NOT NULL
GROUP BY 1 ORDER BY 1"""
try:
    result_df = pd.read_sql_query(text(query_sql), engine)
    display(result_df.head(100))
    print(f"Số dòng kết quả: {len(result_df):,}")
except Exception as e:
    print("Truy vấn chưa chạy được. Kiểm tra kết nối Supabase, tên bảng/cột và kiểu dữ liệu.")
    raise


### Q2 — Số đơn theo tháng

**Mục đích:** dùng kết quả để trả lời câu hỏi nghiệp vụ số 2. Kiểm tra tên bảng/cột nếu schema thực tế khác bộ dữ liệu Olist chuẩn.

In [ ]:
# Câu hỏi 2: Q2 — Số đơn theo tháng
query_sql = """SELECT date_trunc('month', order_purchase_timestamp)::date AS month,
       count(*) AS orders
FROM olist_orders_dataset
WHERE order_purchase_timestamp IS NOT NULL
GROUP BY 1 ORDER BY 1"""
try:
    result_df = pd.read_sql_query(text(query_sql), engine)
    display(result_df.head(100))
    print(f"Số dòng kết quả: {len(result_df):,}")
except Exception as e:
    print("Truy vấn chưa chạy được. Kiểm tra kết nối Supabase, tên bảng/cột và kiểu dữ liệu.")
    raise


### Q3 — Danh mục bán chạy

**Mục đích:** dùng kết quả để trả lời câu hỏi nghiệp vụ số 3. Kiểm tra tên bảng/cột nếu schema thực tế khác bộ dữ liệu Olist chuẩn.

In [ ]:
# Câu hỏi 3: Q3 — Danh mục bán chạy
query_sql = """SELECT coalesce(p.product_category_name,'unknown') category,
       count(*) item_lines, count(distinct oi.order_id) orders,
       round(sum(oi.price)::numeric,2) merchandise_revenue_brl
FROM olist_order_items_dataset oi
JOIN olist_orders_dataset o USING(order_id)
JOIN olist_products_dataset p USING(product_id)
WHERE o.order_status NOT IN ('canceled','unavailable')
GROUP BY 1 ORDER BY item_lines DESC LIMIT 30"""
try:
    result_df = pd.read_sql_query(text(query_sql), engine)
    display(result_df.head(100))
    print(f"Số dòng kết quả: {len(result_df):,}")
except Exception as e:
    print("Truy vấn chưa chạy được. Kiểm tra kết nối Supabase, tên bảng/cột và kiểu dữ liệu.")
    raise


### Q4 — AOV theo phương thức thanh toán

**Mục đích:** dùng kết quả để trả lời câu hỏi nghiệp vụ số 4. Kiểm tra tên bảng/cột nếu schema thực tế khác bộ dữ liệu Olist chuẩn.

In [ ]:
# Câu hỏi 4: Q4 — AOV theo phương thức thanh toán
query_sql = """WITH op AS (SELECT order_id, payment_type, sum(payment_value) amount
            FROM olist_order_payments_dataset GROUP BY order_id,payment_type),
valid AS (SELECT order_id FROM olist_orders_dataset
          WHERE order_status NOT IN ('canceled','unavailable'))
SELECT op.payment_type, count(distinct op.order_id) orders,
       round(avg(op.amount)::numeric,2) avg_payment_per_order_brl,
       round(sum(op.amount)::numeric,2) payment_total_brl
FROM op JOIN valid USING(order_id)
GROUP BY op.payment_type ORDER BY orders DESC"""
try:
    result_df = pd.read_sql_query(text(query_sql), engine)
    display(result_df.head(100))
    print(f"Số dòng kết quả: {len(result_df):,}")
except Exception as e:
    print("Truy vấn chưa chạy được. Kiểm tra kết nối Supabase, tên bảng/cột và kiểu dữ liệu.")
    raise


### Q5 — Seller hoạt động theo quý

**Mục đích:** dùng kết quả để trả lời câu hỏi nghiệp vụ số 5. Kiểm tra tên bảng/cột nếu schema thực tế khác bộ dữ liệu Olist chuẩn.

In [ ]:
# Câu hỏi 5: Q5 — Seller hoạt động theo quý
query_sql = """SELECT date_trunc('quarter',o.order_purchase_timestamp)::date quarter,
       count(distinct oi.seller_id) active_sellers
FROM olist_orders_dataset o JOIN olist_order_items_dataset oi USING(order_id)
WHERE o.order_status NOT IN ('canceled','unavailable')
  AND o.order_purchase_timestamp IS NOT NULL
GROUP BY 1 ORDER BY 1"""
try:
    result_df = pd.read_sql_query(text(query_sql), engine)
    display(result_df.head(100))
    print(f"Số dòng kết quả: {len(result_df):,}")
except Exception as e:
    print("Truy vấn chưa chạy được. Kiểm tra kết nối Supabase, tên bảng/cột và kiểu dữ liệu.")
    raise


### Q6 — Phân phối review score

**Mục đích:** dùng kết quả để trả lời câu hỏi nghiệp vụ số 6. Kiểm tra tên bảng/cột nếu schema thực tế khác bộ dữ liệu Olist chuẩn.

In [ ]:
# Câu hỏi 6: Q6 — Phân phối review score
query_sql = """SELECT review_score, count(*) review_rows, count(distinct order_id) orders
FROM olist_order_reviews_dataset GROUP BY review_score ORDER BY review_score"""
try:
    result_df = pd.read_sql_query(text(query_sql), engine)
    display(result_df.head(100))
    print(f"Số dòng kết quả: {len(result_df):,}")
except Exception as e:
    print("Truy vấn chưa chạy được. Kiểm tra kết nối Supabase, tên bảng/cột và kiểu dữ liệu.")
    raise


### Q7 — Khách mua lại và tỷ trọng doanh thu

**Mục đích:** dùng kết quả để trả lời câu hỏi nghiệp vụ số 7. Kiểm tra tên bảng/cột nếu schema thực tế khác bộ dữ liệu Olist chuẩn.

In [ ]:
# Câu hỏi 7: Q7 — Khách mua lại và tỷ trọng doanh thu
query_sql = """WITH rev AS (SELECT order_id,sum(payment_value) amount FROM olist_order_payments_dataset GROUP BY order_id),
co AS (SELECT c.customer_unique_id,o.order_id,coalesce(rev.amount,0) amount
       FROM olist_customers_dataset c JOIN olist_orders_dataset o USING(customer_id)
       LEFT JOIN rev USING(order_id)
       WHERE o.order_status NOT IN ('canceled','unavailable')),
cs AS (SELECT customer_unique_id,count(distinct order_id) n_orders,sum(amount) spend
       FROM co GROUP BY customer_unique_id)
SELECT count(*) customers,
       count(*) FILTER(WHERE n_orders>1) repeat_customers,
       round(100.0*count(*) FILTER(WHERE n_orders>1)/nullif(count(*),0),2) repeat_rate_pct,
       round(100.0*sum(spend) FILTER(WHERE n_orders>1)/nullif(sum(spend),0),2) repeat_revenue_share_pct
FROM cs"""
try:
    result_df = pd.read_sql_query(text(query_sql), engine)
    display(result_df.head(100))
    print(f"Số dòng kết quả: {len(result_df):,}")
except Exception as e:
    print("Truy vấn chưa chạy được. Kiểm tra kết nối Supabase, tên bảng/cột và kiểu dữ liệu.")
    raise


### Q8 — Điểm review trung bình

**Mục đích:** dùng kết quả để trả lời câu hỏi nghiệp vụ số 8. Kiểm tra tên bảng/cột nếu schema thực tế khác bộ dữ liệu Olist chuẩn.

In [ ]:
# Câu hỏi 8: Q8 — Điểm review trung bình
query_sql = """SELECT round(avg(review_score)::numeric,2) avg_review_score,
       count(*) review_rows, count(*) FILTER(WHERE review_score IS NULL) missing_scores
FROM olist_order_reviews_dataset"""
try:
    result_df = pd.read_sql_query(text(query_sql), engine)
    display(result_df.head(100))
    print(f"Số dòng kết quả: {len(result_df):,}")
except Exception as e:
    print("Truy vấn chưa chạy được. Kiểm tra kết nối Supabase, tên bảng/cột và kiểu dữ liệu.")
    raise


### Q9 — Tỷ lệ hủy đơn

**Mục đích:** dùng kết quả để trả lời câu hỏi nghiệp vụ số 9. Kiểm tra tên bảng/cột nếu schema thực tế khác bộ dữ liệu Olist chuẩn.

In [ ]:
# Câu hỏi 9: Q9 — Tỷ lệ hủy đơn
query_sql = """SELECT count(*) all_orders,
       count(*) FILTER(WHERE order_status='canceled') canceled_orders,
       round(100.0*count(*) FILTER(WHERE order_status='canceled')/nullif(count(*),0),2) cancellation_rate_pct
FROM olist_orders_dataset"""
try:
    result_df = pd.read_sql_query(text(query_sql), engine)
    display(result_df.head(100))
    print(f"Số dòng kết quả: {len(result_df):,}")
except Exception as e:
    print("Truy vấn chưa chạy được. Kiểm tra kết nối Supabase, tên bảng/cột và kiểu dữ liệu.")
    raise


### Q10 — Top sản phẩm theo doanh thu hàng hóa

**Mục đích:** dùng kết quả để trả lời câu hỏi nghiệp vụ số 10. Kiểm tra tên bảng/cột nếu schema thực tế khác bộ dữ liệu Olist chuẩn.

In [ ]:
# Câu hỏi 10: Q10 — Top sản phẩm theo doanh thu hàng hóa
query_sql = """SELECT oi.product_id, coalesce(p.product_category_name,'unknown') category,
       count(*) item_lines, count(distinct oi.order_id) orders,
       round(sum(oi.price)::numeric,2) merchandise_revenue_brl
FROM olist_order_items_dataset oi JOIN olist_orders_dataset o USING(order_id)
JOIN olist_products_dataset p USING(product_id)
WHERE o.order_status NOT IN ('canceled','unavailable')
GROUP BY oi.product_id,2 ORDER BY item_lines DESC LIMIT 20"""
try:
    result_df = pd.read_sql_query(text(query_sql), engine)
    display(result_df.head(100))
    print(f"Số dòng kết quả: {len(result_df):,}")
except Exception as e:
    print("Truy vấn chưa chạy được. Kiểm tra kết nối Supabase, tên bảng/cột và kiểu dữ liệu.")
    raise


### Q11 — Phương thức thanh toán

**Mục đích:** dùng kết quả để trả lời câu hỏi nghiệp vụ số 11. Kiểm tra tên bảng/cột nếu schema thực tế khác bộ dữ liệu Olist chuẩn.

In [ ]:
# Câu hỏi 11: Q11 — Phương thức thanh toán
query_sql = """SELECT payment_type, count(distinct order_id) orders,
       round(sum(payment_value)::numeric,2) payment_total_brl
FROM olist_order_payments_dataset GROUP BY payment_type ORDER BY orders DESC"""
try:
    result_df = pd.read_sql_query(text(query_sql), engine)
    display(result_df.head(100))
    print(f"Số dòng kết quả: {len(result_df):,}")
except Exception as e:
    print("Truy vấn chưa chạy được. Kiểm tra kết nối Supabase, tên bảng/cột và kiểu dữ liệu.")
    raise


### Q12 — Review và doanh thu theo điểm

**Mục đích:** dùng kết quả để trả lời câu hỏi nghiệp vụ số 12. Kiểm tra tên bảng/cột nếu schema thực tế khác bộ dữ liệu Olist chuẩn.

In [ ]:
# Câu hỏi 12: Q12 — Review và doanh thu theo điểm
query_sql = """WITH pay AS (SELECT order_id,sum(payment_value) amount FROM olist_order_payments_dataset GROUP BY order_id)
SELECT r.review_score,count(distinct r.order_id) orders,
       round(avg(pay.amount)::numeric,2) avg_order_payment_brl,
       round(sum(pay.amount)::numeric,2) payment_total_brl
FROM olist_order_reviews_dataset r JOIN pay USING(order_id)
JOIN olist_orders_dataset o USING(order_id)
WHERE o.order_status NOT IN ('canceled','unavailable')
GROUP BY r.review_score ORDER BY r.review_score"""
try:
    result_df = pd.read_sql_query(text(query_sql), engine)
    display(result_df.head(100))
    print(f"Số dòng kết quả: {len(result_df):,}")
except Exception as e:
    print("Truy vấn chưa chạy được. Kiểm tra kết nối Supabase, tên bảng/cột và kiểu dữ liệu.")
    raise


### Q13 — Doanh thu hàng hóa và tỷ lệ phí vận chuyển (không phải profit margin)

**Mục đích:** dùng kết quả để trả lời câu hỏi nghiệp vụ số 13. Kiểm tra tên bảng/cột nếu schema thực tế khác bộ dữ liệu Olist chuẩn.

In [ ]:
# Câu hỏi 13: Q13 — Doanh thu hàng hóa và tỷ lệ phí vận chuyển (không phải profit margin)
query_sql = """SELECT coalesce(p.product_category_name,'unknown') category,
       round(sum(oi.price)::numeric,2) merchandise_revenue_brl,
       round(sum(oi.freight_value)::numeric,2) freight_brl,
       round(100.0*sum(oi.freight_value)/nullif(sum(oi.price),0)::numeric,2) freight_to_item_revenue_pct
FROM olist_order_items_dataset oi JOIN olist_products_dataset p USING(product_id)
JOIN olist_orders_dataset o USING(order_id)
WHERE o.order_status NOT IN ('canceled','unavailable')
GROUP BY 1 ORDER BY merchandise_revenue_brl DESC"""
try:
    result_df = pd.read_sql_query(text(query_sql), engine)
    display(result_df.head(100))
    print(f"Số dòng kết quả: {len(result_df):,}")
except Exception as e:
    print("Truy vấn chưa chạy được. Kiểm tra kết nối Supabase, tên bảng/cột và kiểu dữ liệu.")
    raise


### Q14 — Retention theo bang khách hàng

**Mục đích:** dùng kết quả để trả lời câu hỏi nghiệp vụ số 14. Kiểm tra tên bảng/cột nếu schema thực tế khác bộ dữ liệu Olist chuẩn.

In [ ]:
# Câu hỏi 14: Q14 — Retention theo bang khách hàng
query_sql = """WITH x AS (SELECT c.customer_unique_id,c.customer_state,count(distinct o.order_id) n_orders
           FROM olist_customers_dataset c JOIN olist_orders_dataset o USING(customer_id)
           WHERE o.order_status NOT IN ('canceled','unavailable')
           GROUP BY c.customer_unique_id,c.customer_state)
SELECT customer_state,count(*) customers,
       count(*) FILTER(WHERE n_orders>1) repeat_customers,
       round(100.0*count(*) FILTER(WHERE n_orders>1)/nullif(count(*),0),2) retention_rate_pct
FROM x GROUP BY customer_state ORDER BY customers DESC"""
try:
    result_df = pd.read_sql_query(text(query_sql), engine)
    display(result_df.head(100))
    print(f"Số dòng kết quả: {len(result_df):,}")
except Exception as e:
    print("Truy vấn chưa chạy được. Kiểm tra kết nối Supabase, tên bảng/cột và kiểu dữ liệu.")
    raise


## 4. Chuẩn bị bộ dữ liệu phân tích cho Machine Learning
Các ô dưới đây truy vấn dữ liệu gốc và tạo bảng trung gian trong bộ nhớ Colab. Chúng không sửa hoặc ghi đè bảng Supabase. Mặc định chỉ tính đơn không ở trạng thái `canceled` hoặc `unavailable` cho doanh thu/đặc trưng mua hàng.

In [ ]:
monthly_sql = """
SELECT date_trunc('month', o.order_purchase_timestamp)::date AS month,
       sum(p.payment_value)::float AS revenue
FROM olist_orders_dataset o
JOIN (SELECT order_id, sum(payment_value) payment_value
      FROM olist_order_payments_dataset GROUP BY order_id) p USING(order_id)
WHERE o.order_status NOT IN ('canceled','unavailable')
  AND o.order_purchase_timestamp IS NOT NULL
GROUP BY 1 ORDER BY 1
"""
monthly = pd.read_sql_query(text(monthly_sql), engine, parse_dates=['month'])
monthly = monthly.dropna().sort_values('month').reset_index(drop=True)
print('Số tháng dữ liệu:', len(monthly), '| khoảng:', monthly['month'].min(), '→', monthly['month'].max())
display(monthly.tail())

## 5. Mô hình 1 — Dự đoán khách hàng có mua lại

**Định nghĩa mục tiêu:** với mỗi `customer_unique_id`, dùng thông tin từ đơn hợp lệ đầu tiên để dự đoán khách hàng có phát sinh đơn hợp lệ sau đơn đầu tiên trong khoảng thời gian còn lại của dataset hay không. Đây là bài toán nhãn hồi cứu trong phạm vi dataset, không đảm bảo dự đoán hành vi hiện tại ngoài đời.

**Chống rò rỉ dữ liệu:** đặc trưng chỉ lấy từ đơn đầu tiên; chia train/test theo thời gian của đơn đầu tiên, không chia ngẫu nhiên. Dataset Olist có tỷ lệ khách mua lại thấp nên cần xem precision/recall/F1, không chỉ accuracy.

In [ ]:
customer_feature_sql = """
WITH valid_orders AS (
  SELECT o.order_id, o.customer_id, o.order_purchase_timestamp,
         c.customer_unique_id, c.customer_state,
         row_number() OVER(PARTITION BY c.customer_unique_id ORDER BY o.order_purchase_timestamp,o.order_id) AS rn
  FROM olist_orders_dataset o JOIN olist_customers_dataset c USING(customer_id)
  WHERE o.order_status NOT IN ('canceled','unavailable')
    AND o.order_purchase_timestamp IS NOT NULL
),
first_order AS (
  SELECT * FROM valid_orders WHERE rn=1
),
order_value AS (
  SELECT order_id, sum(payment_value)::float AS payment_value
  FROM olist_order_payments_dataset GROUP BY order_id
),
item_features AS (
  SELECT order_id, count(*)::float AS item_count,
         sum(price)::float AS item_value,
         mode() WITHIN GROUP (ORDER BY product_id) AS mode_product_id
  FROM olist_order_items_dataset GROUP BY order_id
),
review_features AS (
  SELECT order_id, avg(review_score)::float AS review_score
  FROM olist_order_reviews_dataset GROUP BY order_id
),
repeat_label AS (
  SELECT f.customer_unique_id,
         CASE WHEN EXISTS (
           SELECT 1 FROM valid_orders later
           WHERE later.customer_unique_id=f.customer_unique_id
             AND later.order_purchase_timestamp > f.order_purchase_timestamp
         ) THEN 1 ELSE 0 END AS repeat_label
  FROM first_order f
)
SELECT f.customer_unique_id, f.order_id, f.order_purchase_timestamp AS first_order_date,
       f.customer_state, coalesce(v.payment_value,0) AS first_payment_value,
       coalesce(i.item_count,0) AS first_item_count,
       coalesce(i.item_value,0) AS first_item_value,
       r.review_score, extract(month from f.order_purchase_timestamp)::int AS first_order_month,
       extract(year from f.order_purchase_timestamp)::int AS first_order_year,
       l.repeat_label
FROM first_order f
LEFT JOIN order_value v USING(order_id)
LEFT JOIN item_features i USING(order_id)
LEFT JOIN review_features r USING(order_id)
JOIN repeat_label l USING(customer_unique_id)
"""
customer_ml = pd.read_sql_query(text(customer_feature_sql), engine, parse_dates=['first_order_date'])
customer_ml = customer_ml.drop_duplicates('customer_unique_id').sort_values('first_order_date').reset_index(drop=True)
print('Khách hàng:', len(customer_ml), '| tỷ lệ label mua lại:', round(customer_ml.repeat_label.mean(),4))
display(customer_ml.head())

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

features = ['customer_state','first_payment_value','first_item_count','first_item_value','review_score','first_order_month','first_order_year']
cat_cols = ['customer_state']
num_cols = [c for c in features if c not in cat_cols]
# Chia theo mốc thời gian: 80% khách đầu tiên làm train, 20% khách đến sau làm test.
split_at = customer_ml['first_order_date'].quantile(0.8)
train = customer_ml[customer_ml.first_order_date < split_at].copy()
test = customer_ml[customer_ml.first_order_date >= split_at].copy()
X_train, y_train = train[features], train['repeat_label']
X_test, y_test = test[features], test['repeat_label']
print('Mốc chia:', split_at, '| train:',len(train),'| test:',len(test))

preprocess = ColumnTransformer([
 ('num', Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]), num_cols),
 ('cat', Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),
                   ('onehot',OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])
repeat_model = Pipeline([
 ('prep',preprocess),
 ('model',RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42,
                                 min_samples_leaf=3, n_jobs=-1))
])
if len(train) and len(test) and y_train.nunique()>1:
    repeat_model.fit(X_train,y_train)
    pred = repeat_model.predict(X_test)
    proba = repeat_model.predict_proba(X_test)[:,1]
    print(classification_report(y_test,pred,zero_division=0))
    print('Confusion matrix:\n',confusion_matrix(y_test,pred))
    if y_test.nunique()>1:
        print('ROC-AUC:',round(roc_auc_score(y_test,proba),4))
        print('Average precision:',round(average_precision_score(y_test,proba),4))
else:
    print('Không đủ lớp hoặc dữ liệu cho train/test theo thời gian. Kiểm tra số lượng dữ liệu và nhãn.')

In [ ]:
# Thử dự đoán xác suất mua lại cho một số khách trong tập test.
if len(test):
    preview = test[['customer_unique_id','first_order_date','first_payment_value','repeat_label']].copy()
    preview['predicted_repeat_probability'] = repeat_model.predict_proba(X_test)[:,1]
    display(preview.sort_values('predicted_repeat_probability',ascending=False).head(10))

## 6. Mô hình 2 — Dự báo doanh thu theo tháng
Dùng các đặc trưng trễ (lag 1–3 tháng) và trung bình trượt. Giữ lại 6 tháng cuối để đánh giá nếu dữ liệu đủ dài. Dự báo là ước lượng thử nghiệm; chuỗi Olist ngắn và có các tháng thiếu/đột biến nên cần diễn giải thận trọng.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

if len(monthly) < 12:
    print('Có ít hơn 12 tháng dữ liệu; chưa nên kết luận mô hình dự báo ổn định.')
else:
    # Điền các tháng thiếu bằng 0 để có chuỗi tháng liên tục.
    monthly_ts = monthly.set_index('month')['revenue'].asfreq('MS')
    monthly_ts.index = monthly_ts.index.to_period('M').to_timestamp()
    monthly_ts = monthly_ts.fillna(0).astype(float)
    ts = pd.DataFrame({'revenue':monthly_ts})
    for lag in [1,2,3,6]: ts[f'lag_{lag}'] = ts['revenue'].shift(lag)
    ts['rolling3'] = ts['revenue'].shift(1).rolling(3).mean()
    ts['month_num'] = ts.index.month
    ts['year_num'] = ts.index.year
    ts = ts.dropna()
    if len(ts) >= 12:
        test_n = min(6, max(2, len(ts)//5))
        train_ts, test_ts = ts.iloc[:-test_n], ts.iloc[-test_n:]
        reg_features = [c for c in ts.columns if c!='revenue']
        revenue_model = RandomForestRegressor(n_estimators=400,min_samples_leaf=2,random_state=42)
        revenue_model.fit(train_ts[reg_features],train_ts['revenue'])
        test_pred = revenue_model.predict(test_ts[reg_features])
        mae = mean_absolute_error(test_ts['revenue'],test_pred)
        rmse = mean_squared_error(test_ts['revenue'],test_pred)**0.5
        print(f'Test {test_n} tháng | MAE={mae:,.2f} BRL | RMSE={rmse:,.2f} BRL')
        plt.figure(figsize=(12,4))
        plt.plot(train_ts.index,train_ts.revenue,label='Train thực tế')
        plt.plot(test_ts.index,test_ts.revenue,label='Test thực tế')
        plt.plot(test_ts.index,test_pred,label='Dự báo test')
        plt.title('Đánh giá dự báo doanh thu theo tháng'); plt.ylabel('BRL'); plt.legend(); plt.grid(True,alpha=.25); plt.show()
    else:
        print('Sau khi tạo lag còn quá ít điểm dữ liệu để đánh giá.')

In [ ]:
# Dự báo 3 tháng kế tiếp theo kiểu recursive (mỗi dự báo được dùng làm lag cho tháng sau).
if 'revenue_model' in globals() and len(ts) >= 12:
    history = monthly_ts.copy()
    future_rows=[]
    for step in range(1,4):
        next_month = (history.index.max() + pd.offsets.MonthBegin(1))
        row = {
          'lag_1': history.iloc[-1], 'lag_2': history.iloc[-2],
          'lag_3': history.iloc[-3], 'lag_6': history.iloc[-6],
          'rolling3': history.iloc[-3:].mean(),
          'month_num': next_month.month, 'year_num': next_month.year
        }
        forecast = max(0.0,float(revenue_model.predict(pd.DataFrame([row],index=[next_month])[reg_features])[0]))
        future_rows.append({'month':next_month,'forecast_revenue_brl':forecast})
        history.loc[next_month] = forecast
    forecast_df = pd.DataFrame(future_rows)
    display(forecast_df)
    plt.figure(figsize=(12,4))
    plt.plot(monthly_ts.index,monthly_ts.values,label='Doanh thu lịch sử')
    plt.plot(forecast_df.month,forecast_df.forecast_revenue_brl,marker='o',label='Dự báo 3 tháng')
    plt.title('Dự báo doanh thu 3 tháng tiếp theo'); plt.ylabel('BRL'); plt.legend(); plt.grid(True,alpha=.25); plt.show()

## 7. Mô hình 3 — Phân khúc khách hàng RFM + K-Means

- **Recency:** số ngày từ lần mua gần nhất đến ngày mua gần nhất có trong dataset.
- **Frequency:** số đơn hợp lệ.
- **Monetary:** tổng tiền thanh toán theo các đơn hợp lệ.

Do Olist là dataset lịch sử, mốc Recency dùng ngày lớn nhất trong dữ liệu, không dùng ngày hiện tại. K-Means cần chuẩn hóa biến; phân cụm không có nhãn đúng/sai nên cần đọc đặc điểm trung bình từng cụm.

In [ ]:
rfm_sql = """
WITH pay AS (SELECT order_id,sum(payment_value)::float AS payment_value
             FROM olist_order_payments_dataset GROUP BY order_id),
base AS (
 SELECT c.customer_unique_id,o.order_id,o.order_purchase_timestamp,
        coalesce(pay.payment_value,0) AS payment_value
 FROM olist_customers_dataset c JOIN olist_orders_dataset o USING(customer_id)
 LEFT JOIN pay USING(order_id)
 WHERE o.order_status NOT IN ('canceled','unavailable')
   AND o.order_purchase_timestamp IS NOT NULL
)
SELECT customer_unique_id,
       max(order_purchase_timestamp) AS last_purchase,
       count(distinct order_id) AS frequency,
       sum(payment_value) AS monetary,
       max(order_purchase_timestamp) AS reference_date
FROM base GROUP BY customer_unique_id
"""
rfm = pd.read_sql_query(text(rfm_sql),engine,parse_dates=['last_purchase','reference_date'])
reference_date = rfm['last_purchase'].max()
rfm['recency'] = (reference_date-rfm['last_purchase']).dt.days
rfm = rfm[['customer_unique_id','recency','frequency','monetary']].replace([np.inf,-np.inf],np.nan).dropna()
rfm = rfm[(rfm['monetary']>=0) & (rfm['frequency']>=1)].copy()
print('Mốc tính Recency:',reference_date,'| khách hàng:',len(rfm))
display(rfm.describe().T)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Log transform giảm độ lệch lớn của frequency/monetary.
rfm_model_data = rfm[['recency','frequency','monetary']].copy()
rfm_model_data['recency'] = np.log1p(rfm_model_data['recency'].clip(lower=0))
rfm_model_data['frequency'] = np.log1p(rfm_model_data['frequency'].clip(lower=0))
rfm_model_data['monetary'] = np.log1p(rfm_model_data['monetary'].clip(lower=0))
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_model_data)
k = 4
kmeans = KMeans(n_clusters=k,random_state=42,n_init=10)
rfm['cluster'] = kmeans.fit_predict(rfm_scaled)
cluster_profile = rfm.groupby('cluster').agg(
    customers=('customer_unique_id','count'),
    recency_days=('recency','median'),
    avg_orders=('frequency','mean'),
    median_spend_brl=('monetary','median'),
    avg_spend_brl=('monetary','mean')
).round(2).sort_values('median_spend_brl',ascending=False)
display(cluster_profile)
if len(rfm)>k:
    print('Silhouette score (cao hơn thường tốt hơn, nhưng cần kết hợp diễn giải):',round(silhouette_score(rfm_scaled,rfm['cluster'],sample_size=min(5000,len(rfm)),random_state=42),4))

In [ ]:
import plotly.express as px
fig = px.scatter(rfm.sample(min(5000,len(rfm)),random_state=42), x='recency', y='monetary', color=rfm.sample(min(5000,len(rfm)),random_state=42)['cluster'].astype(str),
                 size='frequency', hover_data=['customer_unique_id'],
                 title='Phân khúc RFM: Recency vs Monetary', labels={'cluster':'Cụm','recency':'Recency (ngày)','monetary':'Monetary (BRL)'})
fig.show()

## 8. Lưu kết quả ra CSV
File sẽ được tạo trong phiên Colab hiện tại; tải xuống qua Files bên trái hoặc dùng `files.download`. Không ghi dữ liệu cá nhân nhạy cảm ra ngoài khi không cần thiết. Customer IDs trong Olist là mã định danh dataset.

In [ ]:
from pathlib import Path
out_dir = Path('/content/olist_outputs'); out_dir.mkdir(exist_ok=True)
monthly.to_csv(out_dir/'monthly_revenue_source.csv',index=False)
if 'forecast_df' in globals(): forecast_df.to_csv(out_dir/'revenue_forecast_3_months.csv',index=False)
rfm.to_csv(out_dir/'customer_rfm_clusters.csv',index=False)
if 'preview' in globals(): preview.to_csv(out_dir/'repeat_purchase_predictions_sample.csv',index=False)
print('Đã lưu:',*[str(p) for p in out_dir.iterdir()],sep='\n')

## 9. Cách diễn giải và giới hạn
1. Truy vấn SQL phải chạy thành công trên schema thật của bạn trước khi đưa số liệu vào báo cáo.
2. Với Q13, không thể kết luận lợi nhuận/biên lợi nhuận nếu không có giá vốn và chi phí. Tỷ lệ phí vận chuyển/doanh thu chỉ là chỉ báo.
3. Q6/Q8/Q12 mô tả mối liên hệ giữa đánh giá và hiệu quả bán hàng; tương quan không chứng minh quan hệ nhân quả.
4. Nhãn mua lại bị giới hạn bởi thời gian quan sát dataset; khách ở cuối kỳ có ít thời gian để mua lại hơn. Chia dữ liệu theo thời gian giảm rò rỉ nhưng không loại bỏ hoàn toàn vấn đề kiểm duyệt thời gian.
5. Dự báo doanh thu dùng dữ liệu tháng lịch sử; hãy báo cáo MAE/RMSE từ test chứ không chỉ biểu đồ dự báo.
6. K-Means là phân cụm không giám sát; đặt tên cụm (VIP, tiềm năng, ít hoạt động...) sau khi đọc `cluster_profile`, không gán nhãn theo cảm tính.